# Lab 19-01 · The Inverter's Transfer Curve

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import inverter as iv

## Part 1 · Switching threshold

*Predict* V_M for Example 19-01.1 and 19-01.2, then compare with the closed form and with `ngspice -b vtc.cir`.

In [2]:
sym = iv.Inverter(1.2, 0.35, 0.35, 200e-6, 200e-6)
eqw = iv.Inverter(1.2, 0.35, 0.40, 250e-6, 100e-6)
print("symmetric V_M:", round(sym.vm(), 4), "V; equal widths V_M:", round(eqw.vm(), 4), "V (closed form", round(iv.vm_closed_form(1.2, 0.35, 0.40, 2.5), 4), ")")

symmetric V_M: 0.6 V; equal widths V_M: 0.5243 V (closed form 0.5243 )


## Part 2 · Gain and channel-length modulation

*Predict* how the gain at V_M changes when lambda doubles from 0.1 to 0.2 per volt.

In [3]:
for lam in (0.05, 0.1, 0.2):
    inv = iv.Inverter(1.2, 0.35, 0.35, 200e-6, 200e-6, lam, lam)
    print(f"lambda {lam}: gain at V_M {inv.gain(inv.vm(), 1e-8):7.1f}  (2(1 + lambda VDS)/(lambda Vov) = {2 * (1 + lam * 0.6) / (lam * 0.25):7.1f})")

lambda 0.05: gain at V_M  -164.8  (2(1 + lambda VDS)/(lambda Vov) =   164.8)
lambda 0.1: gain at V_M   -84.8  (2(1 + lambda VDS)/(lambda Vov) =    84.8)
lambda 0.2: gain at V_M   -44.8  (2(1 + lambda VDS)/(lambda Vov) =    44.8)


## Part 3 · Noise margins against sizing

*Predict* which margin shrinks as r = kn/kp grows from 0.25 to 4.

In [4]:
rs = np.logspace(np.log10(0.25), np.log10(4), 25)
nml, nmh = [], []
for r in rs:
    a, b = iv.Inverter(1.2, 0.35, 0.35, 100e-6 * r, 100e-6).noise_margins()
    nml.append(a); nmh.append(b)
plt.semilogx(rs, nml, label="NM_L"); plt.semilogx(rs, nmh, label="NM_H"); plt.xlabel("r = kn/kp"); plt.ylabel("noise margin (V)"); plt.legend(); plt.show()
print("at r = 1:", round(nml[12], 4), round(nmh[12], 4), "V")

at r = 1: 0.5375 0.5375 V


## Part 4 · Regeneration

*Predict* how many stages bring a 0.55 V level to a rail.

In [5]:
inv = iv.Inverter(1.2, 0.35, 0.35, 200e-6, 200e-6, 0.05, 0.05)
print([round(x, 4) for x in iv.chain(0.55, inv, 4)])

[0.55, 1.1189, 0.0, 1.2, 0.0]


## Part 5 · Threshold variation

*Predict* the standard deviation of V_M for 25 mV of threshold sigma on each transistor.

In [6]:
rng = np.random.default_rng(19015)
vms = []
for _ in range(10000):
    dn, dp = rng.normal(0, 0.025, 2)
    vms.append(iv.vm_closed_form(1.2, 0.35 + dn, 0.35 + dp, 1.0))
print("sigma of V_M:", round(float(np.std(vms)) * 1e3, 2), "mV; predicted", round(25 / math.sqrt(2), 2), "mV")

sigma of V_M: 17.7 mV; predicted 17.68 mV


## Part 6 · Subthreshold

*Predict* the slope at the midpoint for supplies of 45, 100 and 200 mV with n = 1.4.

In [7]:
for vdd in (0.0453, 0.1, 0.2):
    x = np.linspace(0, vdd, 401)
    y = iv.subthreshold_vtc(vdd, 1.4, x)
    plt.plot(x / vdd, y / vdd, label=f"{vdd * 1e3:.0f} mV")
    print(f"{vdd * 1e3:5.1f} mV: gain {iv.subthreshold_gain(vdd, 1.4):6.2f}")
plt.xlabel("Vin / VDD"); plt.ylabel("Vout / VDD"); plt.legend(); plt.show()

 45.3 mV: gain   1.00
100.0 mV: gain   4.23
200.0 mV: gain  33.47


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).